# Ablation 2026-09-01 — FlowMatching + Encoder + Temp+Liq

**Run:** `abl_fm_enc_sdf`  
**W&B name:** `1_Sep_2026_fm_enc_sdf`  
**Model:** FlowMatchingModel (Euler ODE sampler)  
**Encoder:** RRDB warm-start (implicit conditioning)  
**Fields:** `temperature` + `liqlabel`

In [ ]:
# ── USER CONFIG — edit paths before running ────────────────────────────────────
RUN_NAME        = 'abl_fm_enc_sdf'
WANDB_RUN_NAME  = '1_Sep_2026_fm_enc_sdf'

# Checkpoint dirs (model on scratch; encoder + eval on group storage)
FLOW_RUN_DIR = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/abl_fm_enc_sdf'
ENC_RUN_DIR  = '/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/enc_sdf'
DATA_ROOT    = '/trace/group/forgelab/ngng/multifield/data_fields'
EVAL_OUT_DIR = '/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/abl_fm_enc_sdf'

# Dataset / model config — must match fm_enc_sdf.yml
FIELD_NAMES      = ['temperature', 'liqlabel']
N_STEPS          = 3
DOWNSCALE_METHOD = 'direct'
NORMALIZE        = 'standardize'
TIMESTEPS        = 1000
SCHEDULE         = 'linear'
FM_N_STEPS       = 50
ENCODING         = True
CONDITIONING     = 'implicit'
DEVICE           = 'cuda'

# Visualisation config
BATCH_INDEX  = 0
SAMPLE_INDEX = 0
BATCH_SIZE   = 4
T_LIQ        = 1700.0
LIQ_THR      = 0.5
MELT_THRESHOLD = 1900.0
ANALYSIS_CH  = 0
ANALYSIS_MAX_BATCH = None

In [ ]:
%matplotlib inline
import os, sys, time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter as _gf
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display as _ipy_display
_plt_show_orig = plt.show
def _show(*a, **kw):
    for n in plt.get_fignums(): _ipy_display(plt.figure(n))
    plt.close('all')
plt.show = _show

if not torch.cuda.is_available() and DEVICE == 'cuda':
    print('No GPU — falling back to CPU'); DEVICE = 'cpu'

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('Could not find project root')
PROJECT_ROOT = find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT} | device: {DEVICE}')

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile
from scipy.spatial import KDTree
from scipy.ndimage import binary_erosion

def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)
def mae_rmse(p, g):
    p, g = np.asarray(p).ravel(), np.asarray(g).ravel()
    return {'MAE': float(np.mean(np.abs(p-g))), 'RMSE': float(np.sqrt(np.mean((p-g)**2)))}

def boundary_pixels(mask): return np.argwhere(mask & ~binary_erosion(mask, structure=np.ones((3,3))))
def chamfer_dist(a, b):
    ba, bb = boundary_pixels(a), boundary_pixels(b)
    if len(ba)==0 or len(bb)==0: return float('nan')
    return float((KDTree(bb).query(ba)[0].mean() + KDTree(ba).query(bb)[0].mean()) / 2)
def consistency_metrics(bt, bl):
    inter=(bt&bl).sum(); union=(bt|bl).sum()
    return (inter/union if union>0 else float('nan')), float(np.mean((bt.astype(float)-bl.astype(float))**2)), chamfer_dist(bt, bl)

fn = FIELD_NAMES
has_sdf = 'sdfliqlabel' in fn
has_T   = 'temperature' in fn
has_liq = 'liqlabel' in fn or has_sdf

def _liq_mask(phys):
    if has_sdf:  return phys[fn.index('sdfliqlabel')] < 0
    return phys[fn.index('liqlabel')] > LIQ_THR

def _mk_contour(ax, phys, sigma=1.5):
    if not (has_T and has_liq): return
    try:
        ax.contour(_gf(_liq_mask(phys).T.astype(float), sigma), levels=[0.5],
                   colors=['white'], linewidths=[1.], origin='lower', alpha=0.85)
    except Exception: pass

In [ ]:
from diffusionsr.analysis.analysis_functions import load_encoder
from diffusionsr.runners.train_flow_matching import FlowMatchingModel

kw = dict(downscale_method=DOWNSCALE_METHOD, root_folder=DATA_ROOT,
          normalize=NORMALIZE, n_steps=N_STEPS, field_names=FIELD_NAMES)
train_ds, dev_ds, test_ds = (SimulationXZDataset(split=s, **kw) for s in ['train','dev','test'])
print(f'Fields: {train_ds.field_names}  HR: {train_ds.img_shape}  factor: {train_ds.factor}x')

lr_enc = load_encoder(ENC_RUN_DIR, train_ds)
print(f'Encoder loaded from {ENC_RUN_DIR}')

model = FlowMatchingModel(
    results_folder=FLOW_RUN_DIR, lr_encoder_folder=ENC_RUN_DIR,
    train_dataset=train_ds, dev_dataset=dev_ds, test_dataset=test_ds,
    timesteps=TIMESTEPS, conditioning=CONDITIONING, encoding=ENCODING,
    schedule=SCHEDULE, device=DEVICE, enc_output=False,
)
model.load_saved_model()
print(f'FlowMatchingModel loaded from {FLOW_RUN_DIR}')

In [ ]:
VIZ_CH = (N_STEPS - 1) * len(FIELD_NAMES)   # current-timestep temperature channel
mid_batch = max(0, len(test_ds) // (2 * BATCH_SIZE))

loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
for i, batch in enumerate(loader):
    if i == mid_batch: break
res_s, hr_s, lr_s, ul_s = [x[SAMPLE_INDEX:SAMPLE_INDEX+1] for x in batch[:4]]
x_e = model.compute_x_e(lr_s, ul_s)
with torch.no_grad():
    samps = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE), x_e=x_e,
                               sampler='euler', n_steps=FM_N_STEPS)
pred_phys = test_ds.unscale_data(samps[-1].cpu().numpy()[0], input_type='hr')
hr_phys   = test_ds.unscale_data(as_numpy(hr_s[0]),          input_type='hr')
lr_phys   = test_ds.unscale_data(as_numpy(lr_s[0]),          input_type='lr')
up_phys   = test_ds.unscale_data(as_numpy(ul_s[0]),          input_type='upscaled_lr')
enc_phys  = test_ds.unscale_data(as_numpy(model.lr_enc(lr_s.to(DEVICE).float()).cpu()[0]), input_type='hr')

fig, axes = plt.subplots(1, 5, figsize=(22, 4), dpi=130)
for ax, (ttl, data, phys) in zip(axes, [
        ('LR Input',    lr_phys[VIZ_CH],   lr_phys),
        ('Upscaled LR', up_phys[VIZ_CH],   up_phys),
        ('CNN Encoder', enc_phys[VIZ_CH],  enc_phys),
        ('FlowMatching',pred_phys[VIZ_CH], pred_phys),
        ('Ground Truth',hr_phys[VIZ_CH],   hr_phys)]):
    ax.imshow(data.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
    _mk_contour(ax, phys[VIZ_CH:VIZ_CH+len(FIELD_NAMES)])
    ax.set_title(ttl, fontsize=9); ax.axis('off')
plt.suptitle(f'{RUN_NAME} — test b{mid_batch} s{SAMPLE_INDEX} (current timestep)', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
import os, wandb
api = wandb.Api()
_entity = os.environ.get("WANDB_ENTITY", "ngng-")

fig, ax = plt.subplots(figsize=(9, 4), dpi=130)
plotted = False

# ── Encoder curves (project: RRDN_Encoder) ───────────────────────────────────
enc_wb_name = '1_Sep_2026_encoder_sdf'
enc_runs = list(api.runs(f"{_entity}/RRDN_Encoder",
                          filters={"display_name": enc_wb_name}))
if enc_runs:
    hist = enc_runs[0].history(keys=['train_loss', 'test_loss'], samples=10000, pandas=True)
    for col in ["train_loss", "test_loss"]:
        if col in hist.columns:
            ls = '--' if 'test' in col else '-'
            ax.plot(hist[col].dropna().values, ls=ls, label=f'Enc:{col}', lw=1.5, alpha=0.9)
            plotted = True
else:
    print(f'W&B: no run "{enc_wb_name}" in {_entity}/RRDN_Encoder')

# ── Flow matching curves (project: Flow3D_SuperResolution) ────────────────────
fm_runs = list(api.runs(f"{_entity}/Flow3D_SuperResolution",
                         filters={"display_name": WANDB_RUN_NAME}))
if fm_runs:
    hist = fm_runs[0].history(keys=['train_loss', 'val_loss'], samples=10000, pandas=True)
    for col in ["train_loss", "val_loss"]:
        if col in hist.columns:
            ls = '--' if 'val' in col else '-'
            ax.plot(hist[col].dropna().values, ls=ls, label=col, lw=1.5, alpha=0.9)
            plotted = True
else:
    print(f'W&B: no run "{WANDB_RUN_NAME}" in {_entity}/Flow3D_SuperResolution')

# ── Fallback: local loss files ────────────────────────────────────────────────
if not plotted:
    from diffusionsr.runners.plot_training_curves import collect_curves
    all_c = [('Enc:'+l, v) for l,v in collect_curves(Path(ENC_RUN_DIR))] + collect_curves(Path(FLOW_RUN_DIR))
    for lbl, vals in all_c:
        ls = '--' if 'val' in lbl.lower() else '-'
        ax.plot(vals, ls=ls, label=lbl, lw=1.5, alpha=0.9)
        plotted = True

if plotted:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'{RUN_NAME} — Training Curves (W&B)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No loss data found in W&B or local files.')

In [ ]:
# ── Full test-set statistics (Euler sampler, FM) ───────────────────────────────
STATS_CH = (N_STEPS - 1) * len(FIELD_NAMES)  # current-timestep temperature channel

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses, mp_errs, kh_errs, vc_maes = [], [], [], [], []
iou_list, cham_list, iou_gt_list, cham_gt_list = [], [], [], []

t0 = time.perf_counter(); n_samples = 0
for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    _, hr_b, lr_b, ul_b = batch[:4]
    xe = model.compute_x_e(lr_b, ul_b)
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                   x_e=xe, sampler='euler', n_steps=FM_N_STEPS)
    pred_last = samps[-1].cpu(); del samps; torch.cuda.empty_cache()
    n_samples += hr_b.shape[0]
    for s in range(hr_b.shape[0]):
        p = test_ds.unscale_data(pred_last.numpy()[s], input_type='hr')
        g = test_ds.unscale_data(as_numpy(hr_b[s]),    input_type='hr')
        p_curr = p[STATS_CH:STATS_CH+len(FIELD_NAMES)]  # current-timestep fields only
        g_curr = g[STATS_CH:STATS_CH+len(FIELD_NAMES)]
        m = mae_rmse(p_curr[0], g_curr[0])
        maes.append(m['MAE']); rmses.append(m['RMSE'])
        try:
            pmp, pkh = get_profile(p_curr[0:1])
            gmp, gkh = get_profile(g_curr[0:1])
            mp_errs.append(float(np.mean(np.abs(pmp-gmp))))
            kh_errs.append(float(np.mean(np.abs(pkh-gkh))))
        except Exception: pass
        vc_maes.append(float(np.mean(np.abs((p_curr[0]>MELT_THRESHOLD).astype(float)
                                           -(g_curr[0]>MELT_THRESHOLD).astype(float)))))
        if has_T and has_liq:
            iou, _, cham  = consistency_metrics(p_curr[0]>T_LIQ, _liq_mask(p_curr))
            ioug,_, chamg = consistency_metrics(g_curr[0]>T_LIQ, _liq_mask(g_curr))
            iou_list.append(iou); cham_list.append(cham)
            iou_gt_list.append(ioug); cham_gt_list.append(chamg)

elapsed = time.perf_counter() - t0
print(f'Test-set stats (n={len(maes)}, Euler n_steps={FM_N_STEPS}):')
print(f'  MAE  = {np.nanmean(maes):.4f} ± {np.nanstd(maes):.4f}')
print(f'  RMSE = {np.nanmean(rmses):.4f} ± {np.nanstd(rmses):.4f}')
if mp_errs: print(f'  MP-MAE = {np.nanmean(mp_errs):.2f} ± {np.nanstd(mp_errs):.2f} px')
if kh_errs: print(f'  KH-MAE = {np.nanmean(kh_errs):.2f} ± {np.nanstd(kh_errs):.2f} px')
if vc_maes: print(f'  VC-MAE = {np.nanmean(vc_maes):.4f} ± {np.nanstd(vc_maes):.4f}')
if iou_list:
    print(f'  IOU  pred={np.nanmean(iou_list):.4f}  GT={np.nanmean(iou_gt_list):.4f}')
    print(f'  Cham pred={np.nanmean(cham_list):.2f} px  GT={np.nanmean(cham_gt_list):.2f} px')
print(f'  Speed: {n_samples/elapsed:.2f} samples/s  ({elapsed:.1f}s total)')

In [ ]:
# ── Multi-sample grid ────────────────────────────────────────────────────────
VIZ_CH = (N_STEPS - 1) * len(FIELD_NAMES)
STATS_CH = VIZ_CH
N_GRID = 5
_gl_mid = max(0, len(test_ds) // (2 * N_GRID))
_gl = DataLoader(test_ds, batch_size=N_GRID, shuffle=False, drop_last=True)
for _gi, _gb in enumerate(_gl):
    if _gi == _gl_mid: break
_res_g, _hr_g, _lr_g, _ul_g = _gb[:4]
_xe_g = model.compute_x_e(_lr_g, _ul_g)
with torch.no_grad():
    _sg = model.batch_sample(dataset=test_ds, batch=_hr_g.to(DEVICE), x_e=_xe_g,
                             sampler='euler', n_steps=FM_N_STEPS)
    _ec_g = model.lr_enc(_lr_g.to(DEVICE).float()).cpu()
_COLS = ['LR Input','Upscaled LR','CNN Encoder','FlowMatching','GT']
fig, axes = plt.subplots(N_GRID, 5, figsize=(22, 3.5*N_GRID), dpi=90)
if N_GRID == 1: axes = axes[np.newaxis]
for r in range(N_GRID):
    _p = test_ds.unscale_data(_sg[-1].cpu().numpy()[r],  input_type='hr')
    _g = test_ds.unscale_data(as_numpy(_hr_g[r]),        input_type='hr')
    _l = test_ds.unscale_data(as_numpy(_lr_g[r]),        input_type='lr')
    _u = test_ds.unscale_data(as_numpy(_ul_g[r]),        input_type='upscaled_lr')
    _e = test_ds.unscale_data(as_numpy(_ec_g[r]),        input_type='hr')
    for c, (data, phys) in enumerate(zip(
            [_l[VIZ_CH], _u[VIZ_CH], _e[VIZ_CH], _p[VIZ_CH], _g[VIZ_CH]],
            [_l, _u, _e, _p, _g])):
        ax = axes[r, c]
        ax.imshow(data.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
        _mk_contour(ax, phys[VIZ_CH:VIZ_CH+len(FIELD_NAMES)]); ax.axis('off')
        if r == 0: ax.set_title(_COLS[c], fontsize=9, fontweight='bold')
plt.suptitle(f'{RUN_NAME} — multi-sample grid (Euler, white=liquid boundary)', fontsize=10)
plt.tight_layout(); plt.show()

# ── Depth profiles ───────────────────────────────────────────────────────────
N_PROF_IN=10; N_PROF_S=5
_mp_preds,_kh_preds,_mp_gt,_kh_gt,_xax=[],[],[],[],None
for _pi,_pb in enumerate(DataLoader(test_ds, batch_size=1, shuffle=False)):
    if _pi>=N_PROF_IN: break
    _,_hrp,_lrp,_ulp = _pb[:4]
    _xep = model.compute_x_e(_lrp, _ulp)
    _gp = test_ds.unscale_data(as_numpy(_hrp[0]), input_type='hr')
    try:
        _gmp,_gkh = get_profile(_gp[VIZ_CH:VIZ_CH+1])
        _mp_gt.append(_gmp); _kh_gt.append(_gkh)
        if _xax is None: _xax = np.arange(len(_gmp))
    except Exception: continue
    _mps,_khs=[],[]
    for _ in range(N_PROF_S):
        with torch.no_grad():
            _ss = model.batch_sample(dataset=test_ds, batch=_hrp.to(DEVICE), x_e=_xep,
                                     sampler='euler', n_steps=FM_N_STEPS)
        _pp = test_ds.unscale_data(_ss[-1].cpu().numpy()[0], input_type='hr')
        try:
            _pmp,_pkh = get_profile(_pp[VIZ_CH:VIZ_CH+1])
            _mps.append(_pmp); _khs.append(_pkh)
        except Exception: pass
    if _mps: _mp_preds.append(np.stack(_mps)); _kh_preds.append(np.stack(_khs))
if _mp_gt and _mp_preds:
    _mp_gt_a=np.stack(_mp_gt); _mp_pd_a=np.concatenate(_mp_preds)
    _kh_gt_a=np.stack(_kh_gt); _kh_pd_a=np.concatenate(_kh_preds)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5),dpi=130)
    for ax,gta,pda,ttl,col in [(ax1,_mp_gt_a,_mp_pd_a,'Melt-Pool Depth','darkorange'),(ax2,_kh_gt_a,_kh_pd_a,'Keyhole Depth','green')]:
        pm,ps=pda.mean(0),pda.std(0); gm,gs=gta.mean(0),gta.std(0)
        ax.plot(_xax,pm,color=col,lw=2,label='FlowMatching (mean)')
        ax.fill_between(_xax,pm-ps,pm+ps,alpha=0.3,color=col,label='±1σ')
        ax.plot(_xax,gm,'k--',lw=1.5,label='GT mean'); ax.fill_between(_xax,gm-gs,gm+gs,alpha=0.15,color='k')
        ax.set_xlabel('x (px)'); ax.set_ylabel('Depth (px)'); ax.set_title(ttl)
        ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
    plt.suptitle(f'{RUN_NAME} — depth profiles',fontsize=11); plt.tight_layout(); plt.show()

# ── Error histograms ──────────────────────────────────────────────────────────
if mp_errs and kh_errs:
    _bl_mp,_bl_kh=[],[]
    for _bb in DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False):
        _,_hrb,_,_ulb=_bb[:4]
        for _si in range(_hrb.shape[0]):
            _ulp=test_ds.unscale_data(as_numpy(_ulb[_si]),input_type='upscaled_lr')
            _hrp=test_ds.unscale_data(as_numpy(_hrb[_si]),input_type='hr')
            try:
                _bmp,_bkh=get_profile(_ulp[VIZ_CH:VIZ_CH+1])
                _gmp2,_gkh2=get_profile(_hrp[VIZ_CH:VIZ_CH+1])
                _bl_mp.append(float(np.mean(np.abs(_bmp-_gmp2)))); _bl_kh.append(float(np.mean(np.abs(_bkh-_gkh2))))
            except Exception: pass
    fig,axes=plt.subplots(1,2,figsize=(13,5),dpi=130)
    for ax,sr,bl,ttl in zip(axes,[mp_errs,kh_errs],[_bl_mp,_bl_kh],['MP Depth Error','KH Depth Error']):
        bmax=max(max(sr+[0.001]),max(bl+[0.001]))*1.1; bins=np.linspace(0,bmax,35)
        ax.hist(sr,bins=bins,alpha=0.75,color='darkorange',label=f'FM μ={np.mean(sr):.2f}px')
        if bl: ax.hist(bl,bins=bins,alpha=0.6,color='gray',label=f'UpLR μ={np.mean(bl):.2f}px')
        ax.axvline(np.mean(sr),color='darkorange',ls='--',lw=2)
        if bl: ax.axvline(np.mean(bl),color='gray',ls='--',lw=2)
        ax.set_xlabel('MAE (px)'); ax.set_ylabel('Count'); ax.set_title(ttl); ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
    plt.suptitle(f'{RUN_NAME} — error histograms',fontsize=11); plt.tight_layout(); plt.show()

# ── Euler step ablation ───────────────────────────────────────────────────────
_ABL_NSTEPS=[5,10,25,50,100]; _ABL_N=4
_ab_mae={n:[] for n in _ABL_NSTEPS}; _ab_mp={n:[] for n in _ABL_NSTEPS}
for _ai,_ab in enumerate(DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)):
    if _ai>=_ABL_N: break
    _,_ahr,_alr,_aul=_ab[:4]; _axe=model.compute_x_e(_alr,_aul)
    for _ns in _ABL_NSTEPS:
        with torch.no_grad():
            _as=model.batch_sample(dataset=test_ds,batch=_ahr.to(DEVICE),x_e=_axe,sampler='euler',n_steps=_ns)
        for _s in range(_ahr.shape[0]):
            _pp=test_ds.unscale_data(_as[-1].cpu().numpy()[_s],input_type='hr')
            _gg=test_ds.unscale_data(as_numpy(_ahr[_s]),input_type='hr')
            _ab_mae[_ns].append(mae_rmse(_pp[STATS_CH],_gg[STATS_CH])['MAE'])
            try:
                _pmp,_=get_profile(_pp[STATS_CH:STATS_CH+1]); _gmp,_=get_profile(_gg[STATS_CH:STATS_CH+1])
                _ab_mp[_ns].append(float(np.mean(np.abs(_pmp-_gmp))))
            except Exception: pass
_mae_v=[np.nanmean(_ab_mae[n]) for n in _ABL_NSTEPS]; _mp_v=[np.nanmean(_ab_mp[n]) if _ab_mp[n] else float('nan') for n in _ABL_NSTEPS]
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5),dpi=130)
ax1.plot(_ABL_NSTEPS,_mae_v,'o-',color='darkorange',lw=2,ms=8); ax1.set_xlabel('Euler steps'); ax1.set_ylabel('MAE'); ax1.set_title('Euler Ablation — Field MAE'); ax1.grid(True,alpha=0.3)
ax2.plot(_ABL_NSTEPS,_mp_v,'s-',color='green',lw=2,ms=8); ax2.set_xlabel('Euler steps'); ax2.set_ylabel('MP-MAE (px)'); ax2.set_title('Euler Ablation — MP Depth'); ax2.grid(True,alpha=0.3)
plt.suptitle(f'{RUN_NAME} — sampler step ablation',fontsize=11); plt.tight_layout(); plt.show()

In [ ]:
# ── Save summary metrics to eval_results ──────────────────────────────────────
import os
os.makedirs(EVAL_OUT_DIR, exist_ok=True)
summary = {
    'run_name': RUN_NAME, 'wandb_run': WANDB_RUN_NAME,
    'model': 'FlowMatching', 'encoding': ENCODING, 'conditioning': CONDITIONING,
    'fields': str(FIELD_NAMES), 'fm_n_steps': FM_N_STEPS,
    'n_test': len(maes),
    'mae_mean': float(np.nanmean(maes)),  'mae_std': float(np.nanstd(maes)),
    'rmse_mean': float(np.nanmean(rmses)),'rmse_std': float(np.nanstd(rmses)),
    'mp_mae_mean': float(np.nanmean(mp_errs)) if mp_errs else float('nan'),
    'mp_mae_std':  float(np.nanstd(mp_errs))  if mp_errs else float('nan'),
    'kh_mae_mean': float(np.nanmean(kh_errs)) if kh_errs else float('nan'),
    'kh_mae_std':  float(np.nanstd(kh_errs))  if kh_errs else float('nan'),
    'vc_mae_mean': float(np.nanmean(vc_maes)) if vc_maes else float('nan'),
    'iou_mean': float(np.nanmean(iou_list)) if iou_list else float('nan'),
    'cham_mean': float(np.nanmean(cham_list)) if cham_list else float('nan'),
}
pd.DataFrame([summary]).to_csv(f'{EVAL_OUT_DIR}/metrics_summary.csv', index=False)
print(f'Saved → {EVAL_OUT_DIR}/metrics_summary.csv')
pd.DataFrame([summary]).T